## Week 1 - Lab 3: Your Digital Twin (Part 1)

Today we build something genuinely useful:

1. A **Gradio chatbot** that represents you and answers questions about your career
2. An **Evaluator** agent that checks each answer meets quality standards before it's returned

This introduces the **Reflection / Self-evaluation** agentic pattern.

> We're not using tools yet — that's Lab 4. Today we focus on the evaluation loop.

## Setup: Add your profile

Before running this notebook:

1. Export your LinkedIn profile as a PDF (`linkedin.pdf`) and place it in the `me/` folder
2. Edit `me/summary.txt` — write 2–3 paragraphs about yourself: your background, skills, what you're looking for
3. Change `name = "Your Name"` in the cell below

If you don't have a LinkedIn profile, just write a short bio in `summary.txt` and skip the PDF step.

In [ ]:
# Packages needed: google-genai, gradio, pypdf
# If missing: uv add gradio pypdf

from dotenv import load_dotenv
from google import genai
from google.genai import types
from pypdf import PdfReader
from pydantic import BaseModel
import gradio as gr
import os

load_dotenv(override=True)
client = genai.Client()

In [ ]:
# *** CHANGE THIS to your name ***
name = "Your Name"

In [ ]:
# Read your LinkedIn PDF (skip if you don't have one)

linkedin = ""
linkedin_path = "me/linkedin.pdf"

if os.path.exists(linkedin_path):
    reader = PdfReader(linkedin_path)
    for page in reader.pages:
        text = page.extract_text()
        if text:
            linkedin += text
    print(f"Loaded LinkedIn profile ({len(linkedin)} chars)")
else:
    print("No linkedin.pdf found — continuing without it")

In [ ]:
# Read your summary

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

print(summary[:300], "...")

In [ ]:
# Build the system prompt — this tells the model who it's playing

system_prompt = (
    f"You are acting as {name}. You answer questions on {name}'s personal website, "
    f"particularly about {name}'s career, background, skills and experience. "
    f"Represent {name} faithfully and professionally — as if talking to a potential employer or client. "
    f"If you don't know the answer, say so honestly.\n\n"
    f"## Summary:\n{summary}\n\n"
)

if linkedin:
    system_prompt += f"## LinkedIn Profile:\n{linkedin}\n\n"

system_prompt += f"With this context, chat with the user, always staying in character as {name}."

print(system_prompt[:400], "...")

In [ ]:
def history_to_contents(history: list[dict]) -> list:
    """Convert Gradio chat history to google-genai Contents format."""
    contents = []
    for msg in history:
        role = "model" if msg["role"] == "assistant" else "user"
        contents.append(
            types.Content(role=role, parts=[types.Part(text=msg["content"])])
        )
    return contents

In [ ]:
def chat(message: str, history: list[dict]) -> str:
    contents = history_to_contents(history)
    contents.append(types.Content(role="user", parts=[types.Part(text=message)]))

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=contents,
        config=types.GenerateContentConfig(system_instruction=system_prompt)
    )
    return response.text

In [ ]:
# Launch the chatbot!
# A browser window will open (or click the link below)

gr.ChatInterface(chat, type="messages").launch()

## Now let's add an Evaluator

The **Reflection** pattern: before returning a reply to the user, run it past a judge.
If the judge rejects it, regenerate with feedback.

This is one of the simplest but most powerful agentic patterns.

In [ ]:
# Pydantic model for structured evaluator output

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [ ]:
evaluator_prompt = (
    f"You evaluate whether a chatbot's reply is acceptable. "
    f"The chatbot is playing the role of {name} on their website. "
    f"It should be professional, accurate, and helpful. "
    f"Reject replies that are off-topic, inaccurate, or break character.\n\n"
    f"## Context about {name}:\n{summary}\n\n"
)
if linkedin:
    evaluator_prompt += f"## LinkedIn:\n{linkedin}\n\n"
evaluator_prompt += "Evaluate the latest reply and respond with JSON: is_acceptable (bool) and feedback (str)."

In [ ]:
def evaluate(reply: str, message: str, history: list) -> Evaluation:
    eval_user_prompt = (
        f"Conversation so far:\n{history}\n\n"
        f"User's latest message: {message}\n\n"
        f"Chatbot's reply: {reply}\n\n"
        "Is this reply acceptable? Provide feedback."
    )
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=eval_user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=evaluator_prompt,
            response_mime_type="application/json",
            response_schema=Evaluation
        )
    )
    return response.parsed

In [ ]:
def rerun(reply: str, message: str, history: list, feedback: str) -> str:
    """Regenerate a reply using the evaluator's feedback."""
    updated_system = (
        system_prompt
        + "\n\n## Quality control rejected your previous reply\n"
        + f"Your rejected reply: {reply}\n\n"
        + f"Reason for rejection: {feedback}\n\n"
        + "Please try again, addressing the feedback."
    )
    contents = history_to_contents(history)
    contents.append(types.Content(role="user", parts=[types.Part(text=message)]))

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=contents,
        config=types.GenerateContentConfig(system_instruction=updated_system)
    )
    return response.text

In [ ]:
def chat_with_evaluation(message: str, history: list[dict]) -> str:
    contents = history_to_contents(history)
    contents.append(types.Content(role="user", parts=[types.Part(text=message)]))

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=contents,
        config=types.GenerateContentConfig(system_instruction=system_prompt)
    )
    reply = response.text

    evaluation = evaluate(reply, message, history)

    if evaluation.is_acceptable:
        print("Evaluator: ✓ Accepted")
    else:
        print(f"Evaluator: ✗ Rejected — {evaluation.feedback}")
        reply = rerun(reply, message, history, evaluation.feedback)

    return reply

In [ ]:
# Quick test — evaluate a single response

test_reply = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Tell me about your work experience?",
    config=types.GenerateContentConfig(system_instruction=system_prompt)
).text

print("Reply:", test_reply[:200])
result = evaluate(test_reply, "Tell me about your work experience?", [])
print("\nEvaluation:", result)

In [ ]:
# Launch chatbot with evaluator
# Watch the terminal for evaluator decisions

gr.ChatInterface(chat_with_evaluation, type="messages").launch()

## What's next

In Lab 4 we'll add **tools** to this chatbot:
- A tool to record visitor email addresses (sent to your phone via Pushover)
- A tool to log questions the bot couldn't answer
- Then we'll deploy it to HuggingFace Spaces

## Exercise

- Update `me/summary.txt` with your real background
- Add your LinkedIn PDF to `me/linkedin.pdf`
- Try asking it something it won't know the answer to — does the evaluator trigger?
- Can you make the evaluator stricter? What criteria would you add?